In [6]:
!git clone https://github.com/ashNotKetchup/SAO-Text-Embedding-Playground

Cloning into 'SAO-Text-Embedding-Playground'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 63 (delta 19), reused 50 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 8.88 MiB | 24.65 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [6]:
!cd SAO-Text-Embedding-Playground/

In [7]:
!pip install -r SAO-Text-Embedding-Playground/requirements.txt

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 57.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
%cd SAO-Text-Embedding-Playground/
!ls

/content/SAO-Text-Embedding-Playground
audio_examples	demo.py    readme.md	     tests
demo.ipynb	functions  requirements.txt


In [5]:
!ls

sample_data


In [5]:
import gradio as gr
import os


from functions.text_gen import text_to_audio
from functions.latent_interpolation import interpolate_audio


def generate_audio(
    text: str,
    model: str,
    interpolation_mode: str = None,
    interpolation_text: str = None,
    interpolation_audio=None,
    interpolation_amount: float = 0.5,
):
    """Generates audio dependent on input"""
    print(
        f"Received inputs - Text: {text}, Model: {model}, Interpolation Mode: {interpolation_mode}, Interpolation Text: {interpolation_text}, Interpolation Audio: {interpolation_audio}, Interpolation Amount: {interpolation_amount}"
    )
    if interpolation_mode == "Text" and interpolation_text:
        print("Performing text-based interpolation (placeholder)")
        interpolation_audio = text_to_audio(interpolation_text)

    if interpolation_audio:
        return interpolate_audio(
            text_to_audio(text), interpolation_audio, model, interpolation_amount
        )

    return interpolate_audio(text_to_audio(text), model_string=model)


def update_interpolation_inputs(mode: str):
    show_text = mode == "Text"
    show_audio = mode == "Audio"
    return (
        gr.update(visible=show_text),
        gr.update(visible=show_audio),
    )


# Interface
def build_interface():
    # determine model choices from "RAVE Models" folder (next to this script)
    models_dir = os.path.join(os.path.dirname(''), "RAVE Models")
    model_choices = []
    try:
        if os.path.isdir(models_dir):
            # list non-hidden files/directories and create full paths
            model_choices = sorted(
                [
                    os.path.join(models_dir, f)
                    for f in os.listdir(models_dir)
                    if not f.startswith(".")
                ]
            )
    except Exception:
        model_choices = []

    # fallback to defaults if folder missing or empty
    if not model_choices:
        model_choices = ["gpt-4o-mini-tts", "gpt-4o-mini", "gpt-4o"]

    with gr.Blocks() as demo:
        gr.Markdown("# Audio Generation Demo")
        with gr.Row():
            with gr.Column():
                txt = gr.Textbox(
                    label="Input text",
                    placeholder="Enter prompt for audio/music generation",
                )
                model = gr.Dropdown(
                    choices=model_choices, value=model_choices[0], label="Model"
                )
                audio = gr.Audio(label="Audio output")
                btn = gr.Button("Generate")
            with gr.Column():
                interp_mode = gr.Dropdown(
                    choices=["Text", "Audio"],
                    value="Text",
                    label="Interpolation source",
                )
                interp_txt = gr.Textbox(
                    label="Interpolation text (optional)",
                    placeholder="Optional text to interpolate from",
                    visible=True,
                )
                interp_audio = gr.Audio(
                    label="Interpolation audio (optional)",
                    sources=["upload"],
                    type="filepath",
                    visible=False,
                )
                interp_amount = gr.Slider(
                    minimum=0.0,
                    maximum=1.0,
                    value=0.5,
                    step=0.05,
                    label="Interpolation amount",
                )

        interp_mode.change(
            fn=update_interpolation_inputs,
            inputs=interp_mode,
            outputs=[interp_txt, interp_audio],
        )
        btn.click(
            fn=generate_audio,
            inputs=[txt, model, interp_mode, interp_txt, interp_audio, interp_amount],
            outputs=audio,
        )

    return demo


if __name__ == "__main__":
    demo = build_interface()
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://35fb4a9f6ddb78f4ed.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
import gradio

In [19]:
!pip install stable_audio_tools

  Using cached alias_free_torch-0.0.6-py3-none-any.whl.metadata (3.8 kB)
  Using cached auraloss-0.4.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached descript_audio_codec-1.0.0-py3-none-any.whl.metadata (7.8 kB)
  Using cached einops_exts-0.0.4-py3-none-any.whl.metadata (621 bytes)
  Using cached ema_pytorch-0.2.3-py3-none-any.whl.metadata (693 bytes)
  Using cached encodec-0.1.1.tar.gz (3.7 MB)
  Preparing metadata (setup.py) ... done
  Using cached importlib_resources-5.12.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached k_diffusion-0.1.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached laion_clap-1.1.4-py3-none-any.whl.metadata (26 kB)
  Using cached local_attention-1.8.6-py3-none-any.whl.metadata (684 bytes)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 44.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.

In [20]:
!pip install pandas

In [21]:
!which python

/usr/local/bin/python
